# 实验八：在 Google Colab 上长轮次训练

本 Notebook 会浅克隆项目，把数据、断点、模型、评估结果和日志持久化到 Google Drive。请先在 Colab 选择 **代码执行程序 → 更改运行时类型 → GPU**，然后按顺序运行。

默认训练：DP-DCGAN 数字 8（30 轮）、DP-CVAE 数字 8（80 轮，ε≈8/ε<2）、DP-WGAN-CP 数字 8（100 轮，ε≈8/ε<2）、DP-CVAE 0–9（60 轮）、DP-ACWGAN-CP 0–9（80 轮）。单张 Colab GPU 应顺序训练，不要并发抢显存。DP-CVAE 与数字 8 DP-WGAN-CP 支持逐轮断点恢复；DP-DCGAN 与 DP-ACWGAN-CP 原训练脚本不支持恢复。

In [ ]:
# 1. 检查 Colab GPU
import os, platform, subprocess, sys
import torch

assert torch.cuda.is_available(), '未检测到 GPU：请在“代码执行程序 → 更改运行时类型”中选择 GPU。'
print('Python:', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('CUDA:', torch.version.cuda)
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
# 2. 挂载 Drive、浅克隆/更新仓库，并安装 Colab 缺少的依赖
from pathlib import Path
from google.colab import drive

drive.mount('/content/drive')
REPO_URL = 'https://github.com/Tomatoisawful/differential-privacy-dcgan.git'
REPO = Path('/content/differential-privacy-dcgan')
if not (REPO / '.git').exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'opacus==1.5.4', 'tqdm>=4.60,<5'], check=True)
os.chdir(REPO)
import opacus, torchvision
print('仓库:', REPO)
print('torch / torchvision / opacus:', torch.__version__, torchvision.__version__, opacus.__version__)

In [ ]:
# 3. 配置持久化目录和运行工具
# 想只跑部分实验时，把对应开关改为 False。已完成模型默认跳过。
DRIVE_ROOT = Path('/content/drive/MyDrive/differential-privacy-dcgan-colab')
DATA_ROOT = DRIVE_ROOT / 'data'
RUN_ROOT = DRIVE_ROOT / 'runs' / 'retrain_long'
FORCE_RETRAIN = False
RUN_DCGAN = True
RUN_DIGIT8_CVAE = True
RUN_DIGIT8_WGAN = True
RUN_ALL_DIGITS_CVAE = True
RUN_ALL_DIGITS_ACWGAN = True
RUN_EVALUATION = True

DATA_ROOT.mkdir(parents=True, exist_ok=True)
RUN_ROOT.mkdir(parents=True, exist_ok=True)

def run_logged(name, command, final_path, resume_path=None):
    final_path = Path(final_path)
    if final_path.exists() and not FORCE_RETRAIN:
        print(f'[{name}] 已完成，跳过：{final_path}')
        return final_path
    command = [str(x) for x in command]
    if resume_path is not None and Path(resume_path).exists() and '--resume' not in command:
        command += ['--resume', str(resume_path)]
        print(f'[{name}] 从断点恢复：{resume_path}')
    log_path = final_path.parent.parent / 'logs' / f'{name}.log'
    log_path.parent.mkdir(parents=True, exist_ok=True)
    mode = 'a' if resume_path is not None and Path(resume_path).exists() else 'w'
    print(f'[{name}] 开始；日志：{log_path}')
    env = os.environ.copy()
    env['PYTHONUNBUFFERED'] = '1'
    with log_path.open(mode, encoding='utf-8') as log:
        log.write('\n$ ' + ' '.join(command) + '\n')
        process = subprocess.Popen(command, cwd=REPO, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
        code = process.wait()
    if code != 0:
        raise RuntimeError(f'{name} 失败（退出码 {code}），请查看 {log_path}')
    if not final_path.exists():
        raise FileNotFoundError(f'{name} 结束但未生成预期模型：{final_path}')
    print(f'[{name}] 完成：{final_path}')
    return final_path

PYTHON = sys.executable
print('Drive 输出目录:', DRIVE_ROOT)

## 4. 数字 8 长训

本单元依次训练必做 DP-DCGAN，以及 ε≈8、ε<2 两组 DP-CVAE / DP-WGAN-CP。DP-DCGAN 的噪声乘数 1.4794921875 用于把 30 轮后的预算维持在 ε<2 附近。

In [ ]:
# 4. 数字 8：30/80/100 轮
if RUN_DCGAN:
    out = RUN_ROOT / 'dp_dcgan_digit8_eps_lt2_e30'
    run_logged('dp_dcgan_digit8_eps_lt2_e30', [PYTHON, 'src/dp_dcgan/train_dp_dcgan.py', '--data-root', DATA_ROOT, '--output-dir', out, '--epochs', 30, '--batch-size', 64, '--target-digit', 8, '--noise-multiplier', 1.4794921875, '--max-grad-norm', 1, '--delta', '1e-5', '--seed', 2026, '--device', 'cuda'], out / 'checkpoints/generator_last.pt')

if RUN_DIGIT8_CVAE:
    for eps_name, eps in [('eps8', 8.0), ('eps1p9', 1.9)]:
        out = RUN_ROOT / f'dp_cvae_digit8_{eps_name}_e80'
        run_logged(f'dp_cvae_digit8_{eps_name}_e80', [PYTHON, 'src/privacy_models/train_dp_vae.py', '--data-root', DATA_ROOT, '--output-dir', out, '--target-digit', 8, '--epochs', 80, '--batch-size', 128, '--max-physical-batch-size', 32, '--grad-sample-mode', 'ghost', '--latent-size', 64, '--features', 32, '--prior-scale', 10, '--learning-rate', 0.001, '--beta', 0.05, '--kl-warmup-epochs', 8, '--ema-decay', 0.99, '--target-epsilon', eps, '--delta', '1e-5', '--max-grad-norm', 1, '--seed', 2026, '--device', 'cuda'], out / 'dp_vae_final.pt', out / 'training_checkpoint.pt')

if RUN_DIGIT8_WGAN:
    for eps_name, eps in [('eps8', 8.0), ('eps1p9', 1.9)]:
        out = RUN_ROOT / f'dp_wgan_cp_digit8_{eps_name}_e100'
        run_logged(f'dp_wgan_cp_digit8_{eps_name}_e100', [PYTHON, 'src/privacy_models/train_dp_wgan_cp_digit8.py', '--data-root', DATA_ROOT, '--output-dir', out, '--target-digit', 8, '--epochs', 100, '--batch-size', 128, '--latent-size', 128, '--generator-features', 64, '--critic-features', 32, '--generator-lr', 0.0001, '--critic-lr', 0.00005, '--critic-steps', 3, '--weight-clip', 0.02, '--ema-decay', 0.99, '--target-epsilon', eps, '--delta', '1e-5', '--max-grad-norm', 1, '--seed', 2026, '--device', 'cuda'], out / 'dp_wgan_cp_digit8_final.pt', out / 'training_checkpoint.pt')

## 5. 完整 MNIST 0–9 长训

DP-CVAE 训练 60 轮并支持恢复；DP-ACWGAN-CP 训练 80 轮，随后用公开 LeNet 做不增加隐私成本的生成器后处理。

In [ ]:
# 5. 完整 0-9：60/80 轮
if RUN_ALL_DIGITS_CVAE:
    out = RUN_ROOT / 'dp_cvae_all_eps8_e60'
    run_logged('dp_cvae_all_eps8_e60', [PYTHON, 'src/privacy_models/train_dp_vae.py', '--data-root', DATA_ROOT, '--output-dir', out, '--epochs', 60, '--batch-size', 256, '--max-physical-batch-size', 32, '--grad-sample-mode', 'ghost', '--latent-size', 64, '--features', 32, '--prior-scale', 10, '--learning-rate', 0.001, '--beta', 0.05, '--kl-warmup-epochs', 8, '--ema-decay', 0.99, '--target-epsilon', 8, '--delta', '1e-5', '--max-grad-norm', 1, '--seed', 2026, '--device', 'cuda'], out / 'dp_vae_final.pt', out / 'training_checkpoint.pt')

if RUN_ALL_DIGITS_ACWGAN:
    out = RUN_ROOT / 'dp_acwgan_cp_all_eps8_e80'
    checkpoint = out / 'checkpoints/dp_wgan_last.pt'
    run_logged('dp_acwgan_cp_all_eps8_e80', [PYTHON, 'src/dp_acwgan_cp/train_dp_wgan.py', '--data-root', DATA_ROOT, '--output-dir', out, '--epochs', 80, '--batch-size', 128, '--latent-size', 128, '--generator-features', 64, '--critic-features', 32, '--critic-steps', 3, '--generator-lr', 0.0001, '--critic-lr', 0.0001, '--wasserstein-weight', 0.01, '--critic-class-weight', 1.0, '--generator-class-weight', 2.0, '--weight-clip', 0.02, '--max-grad-norm', 1, '--target-epsilon', 8, '--delta', '1e-5', '--ema-decay', 0.99, '--save-every', 5, '--seed', 2026, '--device', 'cuda'], checkpoint)
    refined = out / 'refined'
    run_logged('dp_acwgan_cp_all_eps8_e80_refine', [PYTHON, 'src/dp_acwgan_cp/refine_dp_wgan.py', '--checkpoint', checkpoint, '--guide-checkpoint', REPO / 'results/evaluators/lenet_mnist_best.pt', '--output-dir', refined, '--steps', 1200, '--batch-size', 256, '--learning-rate', '2e-5', '--anchor-weight', 0.25, '--tv-weight', 0.002, '--augmentation-noise', 0.025, '--seed', 2026, '--device', 'cuda'], refined / 'dp_wgan_refined.pt')

## 6. 统一评估

对已完成模型分别用 LeNet 与 ConvNet 评估；未训练或未完成的模型会自动跳过。每组生成 10,000 张图，结果保存在对应实验的 `evaluation/` 下。

In [ ]:
# 6. 评估所有已完成的长训模型
def evaluate_generic(name, model_type, checkpoint, task, metadata=None):
    checkpoint = Path(checkpoint)
    if not checkpoint.exists():
        print(f'[{name}] 模型不存在，跳过评估：{checkpoint}')
        return
    for evaluator_name in ('lenet', 'convnet'):
        evaluator = REPO / 'results/evaluators' / f'{evaluator_name}_mnist_best.pt'
        eval_out = checkpoint.parent.parent / 'evaluation' / evaluator_name
        command = [PYTHON, 'src/privacy_models/evaluate_generator.py', '--model-type', model_type, '--checkpoint', checkpoint, '--evaluator-checkpoint', evaluator, '--data-root', DATA_ROOT, '--output-dir', eval_out, '--task', task, '--num-samples', 10000, '--num-real', 10000, '--batch-size', 256, '--seed', 2026, '--device', 'cuda']
        if task == 'single':
            command += ['--target-digit', 8]
        if metadata is not None:
            command += ['--metadata-json', metadata]
        run_logged(f'eval_{name}_{evaluator_name}', command, eval_out / 'evaluation_metrics.json')

def evaluate_acwgan(name, checkpoint):
    checkpoint = Path(checkpoint)
    if not checkpoint.exists():
        print(f'[{name}] 模型不存在，跳过评估：{checkpoint}')
        return
    for evaluator_name in ('lenet', 'convnet'):
        evaluator = REPO / 'results/evaluators' / f'{evaluator_name}_mnist_best.pt'
        eval_out = checkpoint.parent.parent / 'evaluation' / evaluator_name
        command = [PYTHON, 'src/dp_acwgan_cp/evaluate_dp_wgan.py', '--checkpoint', checkpoint, '--evaluator-checkpoint', evaluator, '--data-root', DATA_ROOT, '--output-dir', eval_out, '--num-samples', 10000, '--num-real', 10000, '--batch-size', 256, '--seed', 2026, '--device', 'cuda']
        run_logged(f'eval_{name}_{evaluator_name}', command, eval_out / 'evaluation_metrics.json')

if RUN_EVALUATION:
    dcgan = RUN_ROOT / 'dp_dcgan_digit8_eps_lt2_e30'
    evaluate_generic('dp_dcgan_digit8_eps_lt2_e30', 'dp_dcgan', dcgan / 'checkpoints/generator_last.pt', 'single', dcgan / 'summary.json')
    for eps_name in ('eps8', 'eps1p9'):
        out = RUN_ROOT / f'dp_cvae_digit8_{eps_name}_e80'
        evaluate_generic(f'dp_cvae_digit8_{eps_name}_e80', 'dp_vae', out / 'dp_vae_final.pt', 'single')
        out = RUN_ROOT / f'dp_wgan_cp_digit8_{eps_name}_e100'
        evaluate_generic(f'dp_wgan_cp_digit8_{eps_name}_e100', 'dp_wgan_cp', out / 'dp_wgan_cp_digit8_final.pt', 'single')
    out = RUN_ROOT / 'dp_cvae_all_eps8_e60'
    evaluate_generic('dp_cvae_all_eps8_e60', 'dp_vae', out / 'dp_vae_final.pt', 'all')
    out = RUN_ROOT / 'dp_acwgan_cp_all_eps8_e80'
    evaluate_acwgan('dp_acwgan_cp_all_eps8_e80', out / 'refined/dp_wgan_refined.pt')

print('全部选定任务已处理。结果目录：', RUN_ROOT)